# Tarifas de EPM

Abrí `muestras/epm-tarifas-2016-2021-20261007.csv`. Buscaba el precio de enero de 2021 para el estrato 1, por encima del consumo de subsistencia, y si la fila trae municipio.


In [1]:
from pathlib import Path
import pandas as pd

ruta = Path("..") / "muestras" / "epm-tarifas-2016-2021-20261007.csv"
tarifas = pd.read_csv(ruta, dtype=str)
tarifas.shape


(3623, 12)

In [2]:
tarifas.columns.tolist()


['Tipo de Información',
 'Tipo de Tarifa',
 'Tipo de Dato',
 'Rango de Consumo',
 'Nivel',
 'Propiedad Epm',
 'Compartido',
 'Propiedad del Cliente',
 'Punta',
 'Fuera de Punta',
 'Año',
 'Mes']

In [3]:
tarifas["Año"].value_counts(dropna=False).sort_index()


Año
2016     63
2017    724
2018    710
2019    708
2020    708
2021    709
NaN       1
Name: count, dtype: int64

In [4]:
tarifas[tarifas["Año"].isna()]


,Tipo de Información,Tipo de Tarifa,Tipo de Dato,Rango de Consumo,Nivel,Propiedad Epm,Compartido,Propiedad del Cliente,Punta,Fuera de Punta,Año,Mes
655,Monomia,Costo Unitario,"Costo compra: Gm, i",NaN,I,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
corte = tarifas[
    (tarifas["Año"] == "2021")
    & (tarifas["Tipo de Información"] == "Monomia")
    & (tarifas["Nivel"] == "I")
]
corte["Tipo de Tarifa"].value_counts()


Tipo de Tarifa
Residencial       96
Costo Unitario    72
No Residencial    36
Áreas Comunes     24
Name: count, dtype: int64

In [6]:
enero = corte[
    (corte["Mes"] == "Enero")
    & (corte["Tipo de Tarifa"] == "Residencial")
    & (corte["Tipo de Dato"] == "Estrato 1")
    & (corte["Rango de Consumo"] == "Rango > CS")
]
enero[["Tipo de Información", "Tipo de Tarifa", "Tipo de Dato", "Rango de Consumo", "Nivel", "Propiedad Epm", "Año", "Mes"]]


,Tipo de Información,Tipo de Tarifa,Tipo de Dato,Rango de Consumo,Nivel,Propiedad Epm,Año,Mes
1926,Monomia,Residencial,Estrato 1,Rango > CS,I,"$573,05",2021,Enero


In [7]:
pd.to_numeric(enero["Propiedad Epm"], errors="coerce")


1926   NaN
Name: Propiedad Epm, dtype: float64

Encontré 3.623 filas, de 2016 a 2021. En 2021 hay 709. Una fila no trae año ni precio: es un costo unitario, `Costo compra: Gm, i`, y no entra al corte.

De 2021, información Monomia y nivel I, conté 228 filas. El precio no es un número: llega como `$573,05`. Por eso lo leí como texto. Si lo fuerzo a número, pandas lo deja vacío.

La fila que voy a usar en enero es esta: información `Monomia`, tarifa `Residencial`, dato `Estrato 1`, rango `Rango > CS`, nivel `I`, columna `Propiedad Epm`, valor `$573,05`. El mes viene con nombre, `Enero`, no con un número.

No hay columna de municipio ni de `cod_divipola`. El grano es un precio de EPM en un año, un mes, un tipo y un nivel. No se cruza con una ciudad. El precio que cobre la muestra se guarda en la factura, no se busca uniendo esta tabla con la geografía.
